## Baseline IBM Simulation

In this notebook, we generate synthetic data from an individual-based simulation (IBM) of a species’ population cycle, using fixed true parameters for the vital-rate models. We assume a time-homogeneous demographic process, meaning that the kernel function and vital-rate functions do not depend on time.

This simulation framework follows the IBM approach used for monocarp species in Ellner and Rees (2016), based on the code here:
https://github.com/ipmbook/first-edition/blob/master/Rcode/c2/Monocarp%20Calculations.R

### Simulation procedure

* We simulate the IBM for 100 years, starting from an initial population of 500 plants.
* The first 10 years are discarded, since the initial population is arbitrary and has not yet reached the stable population structure generated by the life cycle process.
* Each row of the resulting dataset represents one individual observed over one time interval.
* We then subsample the population to mimic random sampling of individuals.

This subsampling is valid because the demographic process is assumed to be time invariant. Under this assumption, observing an individual over years 4–5 is equivalent to observing another individual over years 67–68.


### IPM construction
Using the true parameters, we construct integral projection models (IPMs). The integral is discretized using the midpoint method, which converts the kernel function into a kernel matrix whose entries represent transition probabilities. These probabilities are obtained from the fitted model parameters. Finally, we compute population-level statistics.


In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.float_format", "{:.3f}".format)
from scipy.stats import norm
from scipy.linalg import eig
import statsmodels.api as sm
import statsmodels.formula.api as smf
from ipm_utils import (
    survival_kernel_glm, 
    fecundity_kernel_glm, 
    mk_K_glm, 
    sigmoid, 
    growth_fn_glm, 
    flowering_fn_glm, 
    survival_fn_glm, 
    seed_fn_glm, 
    new_recruits_fn_glm
    )

np.random.seed(53241986)

In [12]:
# True model parameters
m_par_true = {
    "surv.int": -0.65, #intercept
    "surv.z": 0.75,    #slope

    "flow.int": -18.0,
    "flow.z": 6.9,

    "grow.int": 0.96,
    "grow.z": 0.59,
    "grow.sd": 0.67,

    "rcsz.int": -0.08,
    "rcsz.sd": 0.76,

    "seed.int": 1.00,
    "seed.z": 2.20,

    "p.r": 0.007
}

### IBM Simulations


In [13]:
# NOTE: Increase initial population size from 250 to 500 to prevent extinction 
# from demographic stochasticity
n_years = 101
init_pop_size = 500 
rng = np.random.default_rng(10)

plant_sizes = rng.normal(
    loc = m_par_true['rcsz.int'],
    scale = m_par_true['rcsz.sd'],
    size = init_pop_size
 )

plant_ages = np.zeros(init_pop_size, dtype = int)
mean_age_history = [plant_ages.mean()]
simulation_records = []
recruits_per_year = []
year = 1
while (year != n_years) and (len(plant_sizes) < 1_500_000):
    current_population_size = len(plant_sizes)
    print(f"Year {year}:Current Population Size = {current_population_size}")

    #Flowering
    flowered = rng.binomial(n = 1, p = flowering_fn_glm(plant_sizes, m_par_true))

    #Seeding
    number_flowering = flowered.sum()
    seed_counts = np.full(shape = len(plant_sizes), fill_value = np.nan)
    seed_counts[flowered == 1] = rng.poisson(lam = seed_fn_glm(plant_sizes[flowered == 1],m_par_true))
    total_seeds = np.nansum(seed_counts)

    #New Recruits Establishment
    if number_flowering == 0:
        number_recruits = 0
    else:
        # Each seed has some probability of becoming a new recruit
        number_recruits = rng.binomial(n = int(total_seeds), p = m_par_true["p.r"])
    recruits_per_year.append(number_recruits)

    #New Recruits Sizes
    recruit_sizes = rng.normal(loc = m_par_true["rcsz.int"], scale = m_par_true["rcsz.sd"], size = number_recruits)

    #Survival
    survived = np.full(shape = len(plant_sizes), fill_value = np.nan)

    non_flowering = (flowered == 0) #Flowering is fatal for monocarps
    survival_probabilities = survival_fn_glm(plant_sizes[non_flowering], m_par_true)

    survived[non_flowering] = rng.binomial(n = 1, p = survival_probabilities)


    #Growth for survivors and non-reproducers
    still_alive = ((flowered == 0) & (survived ==1))
    expected_next_size = (m_par_true["grow.int"] + m_par_true["grow.z"] * plant_sizes[still_alive])
    grown_sizes = rng.normal(loc = expected_next_size, scale = m_par_true["grow.sd"])

    next_year_size = np.full(len(plant_sizes), np.nan)
    next_year_size[still_alive] = grown_sizes

    for i in range(len(plant_sizes)):
        simulation_records.append({
            "z": plant_sizes[i],
            "Repr": flowered[i],
            "Seeds": seed_counts[i],
            "Surv": survived[i],
            "z1": next_year_size[i],
            "age": plant_ages[i],
            "alive": still_alive[i],
            "yr": year
        })

    plant_sizes = np.concatenate([recruit_sizes, grown_sizes])
    plant_ages = np.concatenate([np.zeros(number_recruits, dtype=int),plant_ages[still_alive] + 1])
    year += 1

sim_data = pd.DataFrame(simulation_records)

Year 1:Current Population Size = 500
Year 2:Current Population Size = 183
Year 3:Current Population Size = 152
Year 4:Current Population Size = 259
Year 5:Current Population Size = 312
Year 6:Current Population Size = 226
Year 7:Current Population Size = 335
Year 8:Current Population Size = 231
Year 9:Current Population Size = 259
Year 10:Current Population Size = 290
Year 11:Current Population Size = 305
Year 12:Current Population Size = 228
Year 13:Current Population Size = 210
Year 14:Current Population Size = 283
Year 15:Current Population Size = 496
Year 16:Current Population Size = 407
Year 17:Current Population Size = 307
Year 18:Current Population Size = 301
Year 19:Current Population Size = 346
Year 20:Current Population Size = 300
Year 21:Current Population Size = 314
Year 22:Current Population Size = 406
Year 23:Current Population Size = 358
Year 24:Current Population Size = 390
Year 25:Current Population Size = 252
Year 26:Current Population Size = 354
Year 27:Current Popul

In [14]:
sim_data.describe()

,z,Repr,Seeds,Surv,z1,age,yr
count,352846.000,352846.000,13263.000,339583.000,143210.000,352846.000,352846.000
mean,0.494,0.038,2519.246,0.422,1.422,0.741,82.407
std,1.088,0.190,5828.236,0.494,0.874,1.255,18.460
min,-3.651,0.000,37.000,0.000,-2.386,0.000,1.000
25%,-0.295,0.000,843.000,0.000,0.824,0.000,75.000
50%,0.396,0.000,1408.000,0.000,1.430,0.000,89.000
75%,1.234,0.000,2538.000,1.000,2.022,1.000,96.000
max,5.482,1.000,469842.000,1.000,5.482,19.000,100.000


In [15]:
# Discard first 10 years because population has not converged to the stable state 
sim_data = sim_data[sim_data["yr"] > 10].copy()
sim_data["yr"] = sim_data["yr"] - 10

In [ ]:
dataset_sizes = [250,500,750,1000,2000]
dataset_dict = {}

for n in dataset_sizes:
    data_sample = sim_data.sample(n=n, replace=False, random_state=53241986)
    dataset_dict[f"baseline_ibm_data_{n}"] = data_sample
    data_sample.to_csv(f"baseline_ibm_data_{n}.csv", index=False)

In [17]:
vr_func_true = {
    'repr': flowering_fn_glm, 
    'surv': survival_fn_glm,
    'seed': seed_fn_glm,
    'grow': growth_fn_glm,
    'new_recruits_fn': new_recruits_fn_glm
    }

In [18]:
n_big_matrix = 250


In [19]:
L = -2.65
U = 4.5

IPM_true = mk_K(
    n_mesh_points=n_big_matrix,
    m_par=m_par_true,
    lower_size= L,
    upper_size= U,
    vital_rate_functions= vr_func_true
)



In [20]:
eigenvalues_true, eigenvectors_true = np.linalg.eig(IPM_true["K"])
idx = np.argmax(eigenvalues_true.real)
lambda_true = eigenvalues_true[idx].real
eigenvector_true = eigenvectors_true[:, idx].real

print(lambda_true)
print(eigenvector_true.shape)

1.0593068095360045
(250,)
